# 01 — Manda Planet Preparation + NDVI: No Administrative AOI Clip

This notebook prepares four PlanetScope dates for Manda.

## Important spatial rule

- Planet images are **not clipped with the Manda administrative AOI**.
- A common 3 m grid is created from the **union of all four Planet image footprints**.
- Each Planet date keeps its own real valid coverage.
- Missing coverage remains NoData.
- Sentinel-2 clipping/alignment is done later using this Planet grid.
- Final classification is limited to pixels that are valid across all required dates.

## Dates

- 25 January 2026
- 5 March 2026
- 7 April 2026
- 22 April 2026

## Outputs

- Four prepared 4-band Planet images
- Four Planet NDVI images
- Per-date coverage report
- Four-date common-valid coverage report


> Corrected acquisition date: PlanetScope March scene = **5 March 2026**.

## VS Code local-PC version

এই notebook Google Colab বা local PC ব্যবহার করে না।

মূল project path:

```text
D:\Boro Rice Classification
```

Notebook চালানোর আগে `setup_windows.bat` চালিয়ে `Python (Boro Rice Project)` kernel নির্বাচন করুন।


### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:
# CELL 1 — Portable project path for local use and GitHub reproduction

from pathlib import Path
import os

# Recommended: set BORO_PROJECT_ROOT to the local project directory.
# If it is not set, launch Jupyter from the repository root.
PROJECT_ROOT = Path(
    os.environ.get("BORO_PROJECT_ROOT", str(Path.cwd()))
).expanduser().resolve()

DATA_ROOT = PROJECT_ROOT / "Data"
OUTPUTS_ROOT = PROJECT_ROOT / "Outputs"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Data directory was not found: {DATA_ROOT}\n"
        "Set BORO_PROJECT_ROOT or launch Jupyter from the repository root."
    )

OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Outputs root:", OUTPUTS_ROOT)


In [ ]:
# CELL 2 — Import installed local packages

from pathlib import Path
import math

import numpy as np
import pandas as pd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.windows import transform as window_transform
from rasterio.warp import transform_bounds

print("Rasterio:", rasterio.__version__)

In [ ]:
# CELL 3 — Local Manda project paths

STUDY_AREA = "Manda"

RAW_PLANET_ROOT = (
    DATA_ROOT
    / STUDY_AREA
    / "Planet_Raw"
)

# The Planet date folders are directly inside Planet_Raw.
RAW_AREA_ROOT = RAW_PLANET_ROOT

PREPARED_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Prepared_Planet"
)

NDVI_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Planet_NDVI"
)

REPORT_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Preparation_Reports"
)

for folder in [
    PREPARED_DIR,
    NDVI_DIR,
    REPORT_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

DATE_CONFIG = {
    "20260125": {
        "folder_tokens": [
            "20260125",
            "25_jan",
            "25jan",
        ],
        "label": "25_jan",
    },
    "20260305": {
        "folder_tokens": [
            "20260305",
            "5_mar",
            "5_march",
            "5march",
        ],
        "label": "5_march",
    },
    "20260407": {
        "folder_tokens": [
            "20260407",
            "7_apr",
            "7_april",
            "7april",
        ],
        "label": "7_april",
    },
    "20260422": {
        "folder_tokens": [
            "20260422",
            "22_apr",
            "22_april",
            "22april",
        ],
        "label": "22_april",
    },
}

REFERENCE_DATE = "20260125"

MASK_SNOW = True
MASK_SHADOW = True
MASK_HEAVY_HAZE = True
MASK_CLOUD = True
MASK_UNUSABLE = True

NODATA = -9999.0
BLOCK_SIZE = 512

print("Raw Planet:", RAW_PLANET_ROOT)
print("Prepared output:", PREPARED_DIR)
print("NDVI output:", NDVI_DIR)
print("Reports:", REPORT_DIR)


In [ ]:
# CELL 4 — Helper functions

def normalize_name(text):
    return (
        str(text)
        .lower()
        .replace("-", "")
        .replace("_", "")
        .replace(" ", "")
    )


def iter_windows(
    width,
    height,
    block_size=512,
):
    for row_off in range(
        0,
        height,
        block_size,
    ):
        for col_off in range(
            0,
            width,
            block_size,
        ):
            yield Window(
                col_off=col_off,
                row_off=row_off,
                width=min(
                    block_size,
                    width - col_off,
                ),
                height=min(
                    block_size,
                    height - row_off,
                ),
            )


def folder_contains_tiff(folder):
    return any(
        path.is_file()
        and path.suffix.lower()
        in {".tif", ".tiff"}
        for path in folder.iterdir()
    )


def find_raw_folder(tokens):
    normalized_tokens = [
        normalize_name(token)
        for token in tokens
    ]

    matches = []

    # First preference:
    # raw_planet/Manda/date-folder
    if RAW_AREA_ROOT.exists():
        search_root = RAW_AREA_ROOT

        for folder in [
            search_root,
            *search_root.rglob("*"),
        ]:
            if (
                folder.is_dir()
                and folder_contains_tiff(folder)
            ):
                normalized_path = normalize_name(
                    str(folder)
                )

                if any(
                    token in normalized_path
                    for token in normalized_tokens
                ):
                    matches.append(folder)

    # Fallback:
    # any raw_planet path containing both Manda and the date.
    if not matches and RAW_PLANET_ROOT.exists():
        for folder in RAW_PLANET_ROOT.rglob("*"):
            if (
                not folder.is_dir()
                or not folder_contains_tiff(folder)
            ):
                continue

            normalized_path = normalize_name(
                str(folder)
            )

            if (
                "manda" in normalized_path
                and any(
                    token in normalized_path
                    for token in normalized_tokens
                )
            ):
                matches.append(folder)

    matches = sorted(
        set(matches)
    )

    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the Manda raw folder. "
            f"Date tokens={tokens}; "
            f"matches={[str(path) for path in matches]}"
        )

    return matches[0]


def find_sr_and_udm2(folder):
    tiffs = sorted(
        path
        for path in folder.iterdir()
        if (
            path.is_file()
            and path.suffix.lower()
            in {".tif", ".tiff"}
        )
    )

    udm2_candidates = [
        path
        for path in tiffs
        if "udm2" in path.name.lower()
    ]

    sr_candidates = [
        path
        for path in tiffs
        if (
            "udm2" not in path.name.lower()
            and "ndvi" not in path.name.lower()
        )
    ]

    if not udm2_candidates:
        raise FileNotFoundError(
            f"No UDM2 TIFF found in {folder}"
        )

    if not sr_candidates:
        raise FileNotFoundError(
            f"No Planet SR TIFF found in {folder}"
        )

    composite_udm2 = [
        path
        for path in udm2_candidates
        if "composite" in path.name.lower()
    ]

    if composite_udm2:
        udm2_path = max(
            composite_udm2,
            key=lambda path: path.stat().st_size,
        )
    else:
        udm2_path = max(
            udm2_candidates,
            key=lambda path: path.stat().st_size,
        )

    exact_composite = (
        folder
        / "composite.tif"
    )

    if exact_composite.exists():
        sr_path = exact_composite
    else:
        composite_sr = [
            path
            for path in sr_candidates
            if "composite" in path.name.lower()
        ]

        if composite_sr:
            sr_path = max(
                composite_sr,
                key=lambda path: path.stat().st_size,
            )
        else:
            sr_path = max(
                sr_candidates,
                key=lambda path: path.stat().st_size,
            )

    return sr_path, udm2_path


def infer_band_map(count):
    if count == 4:
        return [1, 2, 3, 4]

    if count == 8:
        return [2, 4, 6, 8]

    raise ValueError(
        f"Planet image has {count} bands; expected 4 or 8."
    )


def detect_scale(src):
    nodata = src.nodata

    for _, window in src.block_windows(1):
        data = src.read(
            window=window,
            masked=False,
        ).astype("float32")

        valid = np.isfinite(data)

        if nodata is not None:
            valid &= data != nodata

        valid &= data > 0
        values = data[valid]

        if values.size:
            median_value = float(
                np.median(values)
            )

            return (
                0.0001
                if median_value > 2.0
                else 1.0
            )

    raise ValueError(
        f"No valid Planet values found: {src.name}"
    )


def quality_valid_from_udm2(udm):
    if udm.shape[0] < 8:
        raise ValueError(
            f"UDM2 has {udm.shape[0]} bands; expected at least 8."
        )

    # UDM2 values 0 and 1 are real data.
    # 255 is reserved for areas outside the warped source footprint.
    coverage_valid = np.all(
        udm != 255,
        axis=0,
    )

    snow = udm[1] > 0
    shadow = udm[2] > 0
    heavy_haze = udm[4] > 0
    cloud = udm[5] > 0
    unusable = udm[7] > 0

    valid = coverage_valid.copy()

    if MASK_SNOW:
        valid &= ~snow

    if MASK_SHADOW:
        valid &= ~shadow

    if MASK_HEAVY_HAZE:
        valid &= ~heavy_haze

    if MASK_CLOUD:
        valid &= ~cloud

    if MASK_UNUSABLE:
        valid &= ~unusable

    return valid


def output_profile(
    grid,
    count,
):
    return {
        "driver": "GTiff",
        "width": grid["width"],
        "height": grid["height"],
        "count": count,
        "dtype": "float32",
        "crs": grid["crs"],
        "transform": grid["transform"],
        "nodata": NODATA,
        "compress": "DEFLATE",
        "predictor": 3,
        "tiled": True,
        "blockxsize": BLOCK_SIZE,
        "blockysize": BLOCK_SIZE,
        "BIGTIFF": "IF_SAFER",
    }


In [ ]:
# CELL 5 — Resolve inputs and create a union Planet grid

resolved = {}

for date, config in DATE_CONFIG.items():
    folder = find_raw_folder(
        config["folder_tokens"]
    )

    sr_path, udm2_path = find_sr_and_udm2(
        folder
    )

    resolved[date] = {
        **config,
        "folder": folder,
        "sr": sr_path,
        "udm2": udm2_path,
    }

    print(
        date,
        "→",
        folder.name,
        "| SR:",
        sr_path.name,
        "| UDM2:",
        udm2_path.name,
    )


reference_path = resolved[
    REFERENCE_DATE
]["sr"]

with rasterio.open(
    reference_path
) as reference:
    reference_crs = reference.crs
    reference_transform = (
        reference.transform
    )

    if reference_crs is None:
        raise ValueError(
            "Reference Planet CRS is missing."
        )

    union_left = np.inf
    union_bottom = np.inf
    union_right = -np.inf
    union_top = -np.inf

    footprint_rows = []

    for date, item in resolved.items():
        with rasterio.open(
            item["sr"]
        ) as src:
            if src.crs is None:
                raise ValueError(
                    f"CRS missing: {item['sr']}"
                )

            bounds_in_reference = (
                transform_bounds(
                    src.crs,
                    reference_crs,
                    *src.bounds,
                    densify_pts=21,
                )
            )

            left, bottom, right, top = (
                bounds_in_reference
            )

            union_left = min(
                union_left,
                left,
            )

            union_bottom = min(
                union_bottom,
                bottom,
            )

            union_right = max(
                union_right,
                right,
            )

            union_top = max(
                union_top,
                top,
            )

            footprint_rows.append({
                "date": date,
                "source": str(
                    item["sr"]
                ),
                "source_crs": str(
                    src.crs
                ),
                "source_width": src.width,
                "source_height": src.height,
                "left_on_reference_crs": left,
                "bottom_on_reference_crs": bottom,
                "right_on_reference_crs": right,
                "top_on_reference_crs": top,
            })

    col_a, row_a = (
        ~reference_transform
    ) * (
        union_left,
        union_top,
    )

    col_b, row_b = (
        ~reference_transform
    ) * (
        union_right,
        union_bottom,
    )

    col0 = int(
        math.floor(
            min(col_a, col_b)
        )
    )

    row0 = int(
        math.floor(
            min(row_a, row_b)
        )
    )

    col1 = int(
        math.ceil(
            max(col_a, col_b)
        )
    )

    row1 = int(
        math.ceil(
            max(row_a, row_b)
        )
    )

    union_window = Window(
        col0,
        row0,
        col1 - col0,
        row1 - row0,
    )

    GRID = {
        "crs": reference_crs,
        "transform": window_transform(
            union_window,
            reference_transform,
        ),
        "width": int(
            union_window.width
        ),
        "height": int(
            union_window.height
        ),
    }

planet_footprint_report = pd.DataFrame(
    footprint_rows
)

planet_footprint_report.to_csv(
    REPORT_DIR
    / "Manda_Planet_Source_Footprints.csv",
    index=False,
)

print()
print(
    "Union Planet grid:",
    GRID["width"],
    "x",
    GRID["height"],
)

print("CRS:", GRID["crs"])
print(
    "No administrative AOI clipping was applied."
)


In [ ]:
# CELL 6 — Create prepared Planet images and NDVI

report_rows = []

for date, item in resolved.items():
    label = item["label"]

    prepared_path = (
        PREPARED_DIR
        / (
            f"Manda_{label}"
            "_SR_prepared_full.tif"
        )
    )

    ndvi_path = (
        NDVI_DIR
        / (
            f"Manda_{label}"
            "_NDVI_full.tif"
        )
    )

    valid_pixel_count = 0
    ndvi_sum = 0.0
    ndvi_min = np.inf
    ndvi_max = -np.inf

    with rasterio.open(
        item["sr"]
    ) as sr_src:
        with rasterio.open(
            item["udm2"]
        ) as udm_src:
            selected_bands = infer_band_map(
                sr_src.count
            )

            scale_factor = detect_scale(
                sr_src
            )

            sr_nodata = (
                sr_src.nodata
                if sr_src.nodata is not None
                else 0
            )

            with WarpedVRT(
                sr_src,
                crs=GRID["crs"],
                transform=GRID["transform"],
                width=GRID["width"],
                height=GRID["height"],
                resampling=Resampling.bilinear,
                src_nodata=sr_nodata,
                nodata=NODATA,
                dtype="float32",
            ) as sr_vrt:
                with WarpedVRT(
                    udm_src,
                    crs=GRID["crs"],
                    transform=GRID["transform"],
                    width=GRID["width"],
                    height=GRID["height"],
                    resampling=Resampling.nearest,
                    src_nodata=255,
                    nodata=255,
                    dtype="uint8",
                ) as udm_vrt:
                    with rasterio.open(
                        prepared_path,
                        "w",
                        **output_profile(
                            GRID,
                            4,
                        ),
                    ) as prepared_dst:
                        with rasterio.open(
                            ndvi_path,
                            "w",
                            **output_profile(
                                GRID,
                                1,
                            ),
                        ) as ndvi_dst:
                            descriptions = [
                                "Blue_Surface_Reflectance",
                                "Green_Surface_Reflectance",
                                "Red_Surface_Reflectance",
                                "NIR_Surface_Reflectance",
                            ]

                            for (
                                band_index,
                                description,
                            ) in enumerate(
                                descriptions,
                                start=1,
                            ):
                                prepared_dst.set_band_description(
                                    band_index,
                                    description,
                                )

                            ndvi_dst.set_band_description(
                                1,
                                "NDVI",
                            )

                            prepared_dst.update_tags(
                                study_area=STUDY_AREA,
                                acquisition_date=date,
                                source=str(
                                    item["sr"]
                                ),
                                source_udm2=str(
                                    item["udm2"]
                                ),
                                scale_factor=scale_factor,
                                spatial_extent=(
                                    "Union of four Planet footprints; "
                                    "no administrative AOI clipping"
                                ),
                                mask_method=(
                                    "Per-date UDM2 mask; "
                                    "light haze retained"
                                ),
                            )

                            ndvi_dst.update_tags(
                                study_area=STUDY_AREA,
                                acquisition_date=date,
                                source_prepared=str(
                                    prepared_path
                                ),
                                formula=(
                                    "(NIR-Red)/(NIR+Red)"
                                ),
                            )

                            for window in iter_windows(
                                GRID["width"],
                                GRID["height"],
                                BLOCK_SIZE,
                            ):
                                raw = sr_vrt.read(
                                    selected_bands,
                                    window=window,
                                ).astype(
                                    "float32"
                                )

                                udm = udm_vrt.read(
                                    window=window
                                )

                                reflectance = (
                                    raw
                                    * scale_factor
                                )

                                quality_valid = (
                                    quality_valid_from_udm2(
                                        udm
                                    )
                                )

                                spectral_valid = (
                                    np.all(
                                        np.isfinite(
                                            reflectance
                                        ),
                                        axis=0,
                                    )
                                    & np.all(
                                        raw != NODATA,
                                        axis=0,
                                    )
                                    & np.any(
                                        raw > 0,
                                        axis=0,
                                    )
                                )

                                valid = (
                                    quality_valid
                                    & spectral_valid
                                )

                                prepared = np.full(
                                    reflectance.shape,
                                    NODATA,
                                    dtype="float32",
                                )

                                prepared[:, valid] = (
                                    reflectance[:, valid]
                                )

                                prepared_dst.write(
                                    prepared,
                                    window=window,
                                )

                                red = reflectance[2]
                                nir = reflectance[3]
                                denominator = (
                                    nir + red
                                )

                                ndvi_valid = (
                                    valid
                                    & np.isfinite(
                                        denominator
                                    )
                                    & (
                                        np.abs(
                                            denominator
                                        )
                                        > 1e-8
                                    )
                                )

                                ndvi = np.full(
                                    red.shape,
                                    NODATA,
                                    dtype="float32",
                                )

                                values = (
                                    (
                                        nir[
                                            ndvi_valid
                                        ]
                                        - red[
                                            ndvi_valid
                                        ]
                                    )
                                    / denominator[
                                        ndvi_valid
                                    ]
                                )

                                values = np.clip(
                                    values,
                                    -1.0,
                                    1.0,
                                )

                                ndvi[
                                    ndvi_valid
                                ] = values

                                ndvi_dst.write(
                                    ndvi,
                                    1,
                                    window=window,
                                )

                                valid_pixel_count += int(
                                    ndvi_valid.sum()
                                )

                                if values.size:
                                    ndvi_sum += float(
                                        values.sum(
                                            dtype="float64"
                                        )
                                    )

                                    ndvi_min = min(
                                        ndvi_min,
                                        float(
                                            values.min()
                                        ),
                                    )

                                    ndvi_max = max(
                                        ndvi_max,
                                        float(
                                            values.max()
                                        ),
                                    )

    if valid_pixel_count == 0:
        raise ValueError(
            f"{date}: zero valid pixels. "
            "Check Planet SR and UDM2 inputs."
        )

    grid_pixels = (
        GRID["width"]
        * GRID["height"]
    )

    report_rows.append({
        "date": date,
        "prepared_path": str(
            prepared_path
        ),
        "ndvi_path": str(
            ndvi_path
        ),
        "valid_pixels": (
            valid_pixel_count
        ),
        "valid_percent_of_union_grid": (
            100.0
            * valid_pixel_count
            / grid_pixels
        ),
        "ndvi_min": ndvi_min,
        "ndvi_max": ndvi_max,
        "ndvi_mean": (
            ndvi_sum
            / valid_pixel_count
        ),
        "scale_factor": scale_factor,
    })

    print(
        f"✅ {date}: prepared + NDVI complete | "
        f"valid={valid_pixel_count:,}"
    )

preparation_report = pd.DataFrame(
    report_rows
)

display(
    preparation_report
)

preparation_report.to_csv(
    REPORT_DIR
    / "Manda_Preparation_Per_Date_Report.csv",
    index=False,
)


In [ ]:
# CELL 7 — Four-date common-valid coverage verification

prepared_paths = []

for date, config in DATE_CONFIG.items():
    label = config["label"]

    prepared_paths.append(
        PREPARED_DIR
        / (
            f"Manda_{label}"
            "_SR_prepared_full.tif"
        )
    )

verification_rows = []
common_valid_pixels = 0

with rasterio.open(
    prepared_paths[0]
) as reference:
    reference_grid = (
        reference.crs,
        reference.transform,
        reference.width,
        reference.height,
    )

    sources = [
        rasterio.open(path)
        for path in prepared_paths
    ]

    try:
        for path, src in zip(
            prepared_paths,
            sources,
        ):
            grid = (
                src.crs,
                src.transform,
                src.width,
                src.height,
            )

            if grid != reference_grid:
                raise ValueError(
                    f"Grid mismatch: {path}"
                )

            verification_rows.append({
                "file": str(path),
                "bands": src.count,
                "width": src.width,
                "height": src.height,
                "crs": str(src.crs),
                "nodata": src.nodata,
                "grid_match": True,
            })

        for window in iter_windows(
            reference.width,
            reference.height,
            BLOCK_SIZE,
        ):
            common_valid = np.ones(
                (
                    int(window.height),
                    int(window.width),
                ),
                dtype=bool,
            )

            for src in sources:
                data = src.read(
                    window=window,
                ).astype(
                    "float32"
                )

                valid = np.all(
                    np.isfinite(data),
                    axis=0,
                )

                if src.nodata is not None:
                    valid &= np.all(
                        data != src.nodata,
                        axis=0,
                    )

                common_valid &= valid

            common_valid_pixels += int(
                common_valid.sum()
            )

    finally:
        for src in sources:
            src.close()

total_grid_pixels = (
    reference_grid[2]
    * reference_grid[3]
)

common_report = pd.DataFrame([
    {
        "study_area": STUDY_AREA,
        "grid_definition": (
            "Union of all four Planet footprints"
        ),
        "administrative_AOI_clip": False,
        "common_valid_pixels_4_dates": (
            common_valid_pixels
        ),
        "common_valid_percent_of_union_grid": (
            100.0
            * common_valid_pixels
            / total_grid_pixels
        ),
        "classification_extent_rule": (
            "Only pixels valid in all required dates"
        ),
    }
])

verification_df = pd.DataFrame(
    verification_rows
)

display(
    verification_df
)

display(
    common_report
)

verification_df.to_csv(
    REPORT_DIR
    / "Manda_Prepared_Grid_Verification.csv",
    index=False,
)

common_report.to_csv(
    REPORT_DIR
    / "Manda_4Date_Common_Coverage_Report.csv",
    index=False,
)

if common_valid_pixels == 0:
    raise ValueError(
        "The four Planet dates have zero common valid coverage."
    )

print("✅ MANDA PREPARATION COMPLETE")
print(
    "No Manda administrative AOI clipping was used."
)
print("Prepared images:", PREPARED_DIR)
print("Planet NDVI:", NDVI_DIR)
print(
    "Four-date common valid pixels:",
    f"{common_valid_pixels:,}",
)
